# Day 10 â€” Branching Graphs, Checkpoints & Human-in-the-Loop (hands-on)

Companion notebook to [`../notes.md`](../notes.md). Builds a real 3-way branching support graph with
checkpointing, then a separate graph demonstrating a human-in-the-loop approval gate.

No API key needed â€” same deterministic-node approach as Day 09, so we can focus entirely on the graph
mechanics.

## Part 1 â€” Branching into more than two paths

In [1]:
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages


class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    category: str


def classify_node(state: SupportState) -> dict:
    question = state["messages"][-1].content.lower()
    if "refund" in question or "bill" in question:
        category = "billing"
    elif "error" in question or "bug" in question:
        category = "technical"
    else:
        category = "general"
    return {"category": category}


def route_after_classify(state: SupportState) -> Literal["billing", "technical", "general"]:
    return state["category"]  # this function's return value picks the next node


def billing_node(state: SupportState) -> dict:
    return {"messages": [AIMessage(content="[billing team] Let me check your refund status.")]}


def technical_node(state: SupportState) -> dict:
    return {"messages": [AIMessage(content="[tech team] Let's debug that error together.")]}


def general_node(state: SupportState) -> dict:
    return {"messages": [AIMessage(content="[general] How else can I help?")]}

`add_conditional_edges` is the key addition over Day 09 â€” instead of one fixed next node, a routing
function picks from several.

In [2]:
builder = StateGraph(SupportState)
builder.add_node("classify", classify_node)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)
builder.add_node("general", general_node)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_after_classify,
    {"billing": "billing", "technical": "technical", "general": "general"},
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("general", END)

checkpointer = MemorySaver()
support_graph = builder.compile(checkpointer=checkpointer)

## Checkpoints: every node's completion gets saved

Give the graph a `thread_id` and every checkpoint gets tied to that conversation, so it can be
inspected or resumed later.

In [3]:
config = {"configurable": {"thread_id": "conversation-1"}}

result = support_graph.invoke(
    {"messages": [HumanMessage(content="I want a refund for my last order")], "category": ""},
    config=config,
)

print("Category chosen:", result["category"])
for m in result["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")

Category chosen: billing
  [HumanMessage] I want a refund for my last order
  [AIMessage] [billing team] Let me check your refund status.


In [4]:
history = list(support_graph.get_state_history(config))
print(f"{len(history)} checkpoints saved for this thread\n")
for i, snap in enumerate(reversed(history)):
    print(f"  checkpoint {i}: next node(s) = {snap.next}")

4 checkpoints saved for this thread

  checkpoint 0: next node(s) = ('__start__',)
  checkpoint 1: next node(s) = ('classify',)
  checkpoint 2: next node(s) = ('billing',)
  checkpoint 3: next node(s) = ()


Every one of those checkpoints is a full snapshot of the state at that moment â€” if the process had
crashed right after "classify," it could resume from checkpoint 1 without redoing anything.

## Part 2 â€” A human-in-the-loop approval gate

A separate, smaller graph: an agent decides to issue a refund, but the graph **pauses before the
refund actually runs** and waits for approval.

In [5]:
class RefundState(TypedDict):
    messages: Annotated[list, add_messages]
    amount: int


def agent_node(state: RefundState) -> dict:
    return {"messages": [AIMessage(content="I'll issue a $50 refund.")], "amount": 50}


def refund_node(state: RefundState) -> dict:
    return {"messages": [AIMessage(content=f"Refund of ${state['amount']} processed.")]}


refund_builder = StateGraph(RefundState)
refund_builder.add_node("agent", agent_node)
refund_builder.add_node("refund", refund_node)
refund_builder.add_edge(START, "agent")
refund_builder.add_edge("agent", "refund")
refund_builder.add_edge("refund", END)

# interrupt_before pauses the graph right before the named node runs.
refund_checkpointer = MemorySaver()
refund_graph = refund_builder.compile(checkpointer=refund_checkpointer, interrupt_before=["refund"])

In [6]:
refund_config = {"configurable": {"thread_id": "refund-request-1"}}

state = refund_graph.invoke(
    {"messages": [HumanMessage(content="Please refund my order")], "amount": 0},
    config=refund_config,
)

print("Graph paused. Messages so far:")
for m in state["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")

snapshot = refund_graph.get_state(refund_config)
print("\nNode waiting to run next:", snapshot.next)
print("(the 'refund' node has NOT executed yet -- no money has actually moved)")

Graph paused. Messages so far:
  [HumanMessage] Please refund my order
  [AIMessage] I'll issue a $50 refund.

Node waiting to run next: ('refund',)
(the 'refund' node has NOT executed yet -- no money has actually moved)


### The human approves â€” resume with `invoke(None, ...)`

Passing `None` as the input tells LangGraph "just continue from where this thread paused," using the
same `thread_id`.

In [7]:
final_state = refund_graph.invoke(None, config=refund_config)

for m in final_state["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")

print("\nGraph finished:", refund_graph.get_state(refund_config).next == ())

  [HumanMessage] Please refund my order
  [AIMessage] I'll issue a $50 refund.
  [AIMessage] Refund of $50 processed.

Graph finished: True


### If the human had rejected instead

Rejection doesn't call `invoke(None, ...)` at all â€” instead you'd either leave the thread paused
forever (nothing runs without approval), or call `graph.update_state(refund_config, {...})` to change
the state and route to a cancellation/explanation node instead of `refund`. The key property is the
same either way: **nothing risky happens until a human explicitly resumes it.**

## Try it yourself

- Add a fourth category to `SupportState`'s routing and see the conditional edge pick it correctly.
- Change `interrupt_before=["refund"]` to `interrupt_before=["agent"]` and see how much earlier the
  pause happens.
- Call `refund_graph.get_state_history(refund_config)` after resuming and see the full checkpoint trail
  for the approval flow, the same way Part 1 did for the branching graph.